# finetune-transformer-lm (GPT)

A refresher on **finetune-transformer-lm** — OpenAI's original code for *"Improving Language Understanding by Generative Pre-Training"* (Radford et al., 2018), the paper that introduced **GPT** (GPT‑1). It established the now‑ubiquitous two‑stage recipe: **generative pre‑training** of a Transformer decoder on unlabeled text, followed by **discriminative fine‑tuning** on each downstream task — with minimal task‑specific architecture.

**Domain:** LLM Inference, Training & Optimization  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**What it is.** `finetune-transformer-lm` is the reference TensorFlow implementation behind GPT‑1. It does two things: (1) loads a Transformer decoder LM pre‑trained on the BooksCorpus, and (2) **fine‑tunes** it on a labeled downstream task (classification, entailment, similarity, multiple‑choice QA) by adding a single linear output layer on top of the final token's representation.

**The problem it solves.** Before GPT‑1, each NLP task needed its own bespoke architecture trained from scratch on limited labeled data. GPT‑1's insight: language modeling on a mountain of *unlabeled* text learns general‑purpose features, and almost any supervised task can then be solved by **fine‑tuning that same network** with one extra linear layer. One architecture, many tasks, far less labeled data — the template every later model (GPT‑2/3, BERT‑style fine‑tuning) builds on.

**Two ideas you actually need to remember:**

1. **Task‑agnostic input transformation.** Rather than change the model per task, *change the input*: serialize every structured input into one ordered token sequence using special delimiter tokens (`start`, `delim`, `extract`). The pre‑trained model never sees a new architecture — just a new sentence shape.
2. **Auxiliary LM objective during fine‑tuning.** Fine‑tune on the task loss *plus* a weighted language‑modeling loss: `L = L_task + λ·L_LM`. Keeping the LM objective alive improves generalization and speeds convergence.

**Reach for it when** you want to understand *why* "pre‑train then fine‑tune" became the default, or you're studying the lineage of modern LLMs. **Don't reach for it** to build something today — the code is TensorFlow 1.x and the model is tiny by modern standards. Use 🤗 `transformers` (`Trainer` / PEFT) for real fine‑tuning now; the *concepts* here are exactly what those tools automate.

## 2. Mental Model

**One frozen-shaped network, reshaped inputs, one tiny head bolted on per task.**

```
            PRE-TRAINING (once, on unlabeled text)
   tokens ─► Transformer decoder ─► next-token LM loss   L1 = Σ log P(u_i | u_<i)
                     │
                     ▼  (keep these weights)
            FINE-TUNING (per task, on labeled data)
   structured input ─► [transform to ONE sequence with delimiters]
        e.g. entailment:  <start> premise <delim> hypothesis <extract>
                     │
                     ▼
        same Transformer decoder  ─►  take hidden state of the <extract> token
                     │                         │
            (auxiliary LM head)        (new linear task head W_y)
                     │                         │
                  λ · L_LM     +          L_task           =   L  (minimize)
```

The whole trick: the *architecture* stays put. A new task is just (a) a new way to flatten its inputs into a delimited sequence and (b) a fresh `D→num_classes` linear layer reading the last token. Everything else is inherited from pre‑training.

## 3. Key Concepts

- **Generative pre‑training (L1).** Standard left‑to‑right LM objective on unlabeled text: maximize `Σ log P(u_i | u_{i-k..i-1})`. This is the expensive stage that learns transferable features. GPT‑1 used a 12‑layer decoder‑only Transformer (~117M params).
- **Discriminative fine‑tuning (L2).** Supervised task loss. Feed the transformed sequence, take the final transformer block's activation at the **last token** (`<extract>`), pass it through a new linear layer `W_y`, softmax → task loss.
- **Auxiliary objective (L3 = L2 + λ·L1).** During fine‑tuning, keep optimizing the LM loss too, weighted by `λ` (paper used λ=0.5). Acts as a regularizer and improves convergence — more helpful on larger labeled datasets.
- **Input transformations ("traversal‑style").** Convert structured inputs into a single contiguous sequence so the pre‑trained model needs no architectural change:
  - *Classification:* `<start> text <extract>`
  - *Entailment:* `<start> premise <delim> hypothesis <extract>`
  - *Similarity:* both orderings, processed independently, then summed (order‑invariant).
  - *Multiple choice / QA:* one sequence per answer option; softmax over the per‑option scores.
- **Special tokens.** `start`, `delim`, `extract` are randomly‑initialized embeddings added on top of the BPE vocab; they're learned during fine‑tuning.
- **BPE tokenization.** GPT‑1 uses byte‑pair encoding (~40k merges) — subword units balancing vocab size against sequence length.
- **Decoder‑only / causal.** Like all GPTs: masked self‑attention, each position attends only to the past (contrast with BERT's bidirectional encoder, which arrived a few months later).

## 4. Setup

The original repo is **TensorFlow 1.x** (plus `ftfy`, `spacy`, `tqdm`) and ships a pre‑trained checkpoint:

```bash
# The original (historical) code — TF 1.x, Python 3.6-era:
git clone https://github.com/openai/finetune-transformer-lm
pip install tensorflow==1.15 ftfy spacy tqdm
python train.py --dataset rocstories --desc rocstories --submit --analysis

# The modern equivalent you'd actually use today:
pip install torch transformers datasets
```

This notebook teaches the **mechanics** with tiny, dependency‑free NumPy so it runs anywhere on CPU. The runnable cells below need only `numpy`; a final, **gated** cell sketches the real 🤗 `transformers` fine‑tune (skipped unless you set an env var, so the notebook still executes end‑to‑end).

In [1]:
import os
import numpy as np

print("numpy:", np.__version__)
print("This notebook runs on CPU with numpy only.")
print("Heavy HF fine-tune cell is gated behind RUN_HF_FINETUNE; unset =", 
      os.getenv("RUN_HF_FINETUNE") is None)

numpy: 2.5.0
This notebook runs on CPU with numpy only.
Heavy HF fine-tune cell is gated behind RUN_HF_FINETUNE; unset = True


## 5. Worked Examples

### Example 1 — Input transformations (the actual GPT‑1 trick)

The single most important idea: every task becomes **one delimited token sequence**, so the pre‑trained decoder needs no architectural surgery. Below we reproduce the `transform_*` logic for the three structural shapes the paper handles.

In [2]:
# GPT-1 adds 3 special tokens on top of the BPE vocabulary.
START, DELIM, EXTRACT = "<start>", "<delim>", "<extract>"
words = "the movie was great terrible boring a it sleeps cat dog".split()
vocab = [START, DELIM, EXTRACT] + words
stoi = {w: i for i, w in enumerate(vocab)}
encode = lambda seq: [stoi[w] for w in seq]

# Each task is squeezed into ONE ordered sequence the LM reads left-to-right.
def transform_clf(text):                       # single-text classification
    return [START] + text.split() + [EXTRACT]

def transform_pair(premise, hypothesis):       # entailment / QA: 2 segments
    return [START] + premise.split() + [DELIM] + hypothesis.split() + [EXTRACT]

def transform_choice(context, choices):        # multiple choice -> N sequences
    return [[START] + context.split() + [DELIM] + c.split() + [EXTRACT]
            for c in choices]

ex_clf = transform_clf("the movie was great")
ex_pair = transform_pair("it sleeps", "the cat sleeps")
ex_mc = transform_choice("the dog", ["it sleeps", "it was boring"])

print("vocab size:", len(vocab), "(3 special + words)")
print("\nclassification:")
print("  tokens:", ex_clf)
print("  ids   :", encode(ex_clf))
print("\nentailment (premise <delim> hypothesis):")
print("  tokens:", ex_pair)
print("  ids   :", encode(ex_pair))
print("\nmultiple choice -> one sequence per option:")
for i, seq in enumerate(ex_mc):
    print(f"  option {i}: {seq}  ->  {encode(seq)}")

vocab size: 14 (3 special + words)

classification:
  tokens: ['<start>', 'the', 'movie', 'was', 'great', '<extract>']
  ids   : [0, 3, 4, 5, 6, 2]

entailment (premise <delim> hypothesis):
  tokens: ['<start>', 'it', 'sleeps', '<delim>', 'the', 'cat', 'sleeps', '<extract>']
  ids   : [0, 10, 11, 1, 3, 12, 11, 2]

multiple choice -> one sequence per option:
  option 0: ['<start>', 'the', 'dog', '<delim>', 'it', 'sleeps', '<extract>']  ->  [0, 3, 13, 1, 10, 11, 2]
  option 1: ['<start>', 'the', 'dog', '<delim>', 'it', 'was', 'boring', '<extract>']  ->  [0, 3, 13, 1, 10, 5, 8, 2]


Notice there is **no per‑task model code** — only per‑task *string surgery*. The same network consumes all three shapes; only the final linear head's output dimension differs. For multiple choice, each option becomes its own sequence and the model scores them, then a softmax picks the answer.

### Example 2 — The combined fine‑tuning objective `L = L_clf + λ·L_LM`

The paper's other key idea is keeping the language‑modeling loss alive during fine‑tuning. Here we fine‑tune a tiny shared layer `W` (standing in for the transformer) that feeds **two heads**: a task classifier and an LM head. The shared `W` receives gradient from both, weighted by `λ` — exactly GPT‑1's `L3 = L2 + λ·L1`. We compare `λ=0` (task only) vs `λ=0.5`.

In [3]:
rng = np.random.default_rng(0)

# X = features of the <extract> token out of the transformer.
# y = downstream class label.   t = next-token target (the LM auxiliary signal).
N, D, C, V = 240, 8, 2, 6
X = rng.standard_normal((N, D))
w_true = rng.standard_normal(D)
y = (X @ w_true > 0).astype(int)                 # task labels
t = (X @ rng.standard_normal((D, V))).argmax(1)  # auxiliary LM targets
Xtr, Xte, ytr, yte = X[:40], X[40:], y[:40], y[40:]   # tiny train, big test
ttr = t[:40]

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z); return e / e.sum(1, keepdims=True)

def ce(p, idx):                                  # mean cross-entropy
    return -np.log(p[np.arange(len(idx)), idx] + 1e-9).mean()

def fit(lmbda, steps=600, lr=0.2):
    """Shared layer W feeds a task head Wc AND an LM head Wl.
    Objective = L_clf + lmbda * L_lm  (GPT-1's L3 = L2 + lambda*L1)."""
    W  = np.eye(D) + 0.01 * rng.standard_normal((D, D))
    Wc = 0.01 * rng.standard_normal((D, C))
    Wl = 0.01 * rng.standard_normal((D, V))
    n = len(Xtr)
    for _ in range(steps):
        H  = Xtr @ W                             # shared representation
        Pc, Pl = softmax(H @ Wc), softmax(H @ Wl)
        gc = Pc.copy(); gc[np.arange(n), ytr] -= 1; gc /= n
        gl = Pl.copy(); gl[np.arange(n), ttr] -= 1; gl /= n
        dH = gc @ Wc.T + lmbda * (gl @ Wl.T)     # BOTH heads flow into W
        W  -= lr * (Xtr.T @ dH)
        Wc -= lr * (H.T @ gc)
        Wl -= lr * lmbda * (H.T @ gl)
    clf_loss = ce(softmax((Xtr @ W) @ Wc), ytr)
    lm_loss  = ce(softmax((Xtr @ W) @ Wl), ttr)
    test_acc = (softmax((Xte @ W) @ Wc).argmax(1) == yte).mean()
    return clf_loss, lm_loss, test_acc

print(f"random LM baseline = log(V) = {np.log(V):.3f}\n")
print(f"{'lambda':>7} | {'L_clf':>7} | {'L_lm':>6} | {'test acc':>8}")
print("-" * 38)
for lmbda in (0.0, 0.5):
    lc, ll, acc = fit(lmbda)
    print(f"{lmbda:>7.1f} | {lc:>7.3f} | {ll:>6.3f} | {acc:>8.3f}")

random LM baseline = log(V) = 1.792

 lambda |   L_clf |   L_lm | test acc
--------------------------------------
    0.0 |   0.005 |  1.782 |    0.910
    0.5 |   0.005 |  0.021 |    0.905


With `λ=0` the LM head is never trained, so its loss sits at the random baseline `log(V)≈1.79`. With `λ=0.5` the *same shared layer* is pushed to also model the LM target, driving `L_lm` far down while task accuracy stays comparable — the auxiliary objective shapes the representation at no cost to the task. (In the paper the generalization *benefit* of `λ` grows with dataset size; on this toy, independent signals, it mostly demonstrates the gradient plumbing.)

### Example 3 (gated) — the modern equivalent in 🤗 `transformers`

What `finetune-transformer-lm` did by hand, `transformers` does in a few lines. This cell is **gated** behind an env var (it downloads a model), so the notebook still runs top‑to‑bottom without it. Set `RUN_HF_FINETUNE=1` to actually execute.

In [4]:
if os.getenv("RUN_HF_FINETUNE"):
    from transformers import (AutoTokenizer,
                              AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)
    from datasets import load_dataset

    # GPT-2 with a classification head == GPT-1's fine-tuning recipe, modernized.
    tok = AutoTokenizer.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token
    model = AutoModelForSequenceClassification.from_pretrained(
        "gpt2", num_labels=2)
    model.config.pad_token_id = tok.pad_token_id

    ds = load_dataset("imdb", split="train[:1%]")
    enc = ds.map(lambda b: tok(b["text"], truncation=True, max_length=128),
                 batched=True)
    args = TrainingArguments(output_dir="out", per_device_train_batch_size=8,
                             num_train_epochs=1, report_to=[])
    Trainer(model=model, args=args, train_dataset=enc).train()
    print("done")
else:
    print("skipped: set RUN_HF_FINETUNE=1 to run the real HF fine-tune.")
    print("Shape of the call: AutoModelForSequenceClassification.from_pretrained")
    print("  ('gpt2', num_labels=2)  +  Trainer(...).train()")

skipped: set RUN_HF_FINETUNE=1 to run the real HF fine-tune.
Shape of the call: AutoModelForSequenceClassification.from_pretrained
  ('gpt2', num_labels=2)  +  Trainer(...).train()


## 6. Gotchas & Pitfalls

- **The classifier reads the *last* token, not the first.** Because the model is causal (left‑to‑right), only the final `<extract>` position has attended to the whole sequence. Pooling the first token (BERT's `[CLS]` habit) is wrong for a decoder‑only LM.
- **`pad_token` is unset on GPT‑style tokenizers.** GPT‑1/2 had no padding token. For batched fine‑tuning you must set one (commonly reuse `eos`) *and* tell the model's config — forgetting throws at the first batch (as the gated cell shows).
- **Special tokens must be learned, not assumed.** `start`/`delim`/`extract` start as random embeddings; they only become meaningful *during* fine‑tuning. Reusing a pre‑training token id by accident silently corrupts inputs.
- **`λ` is a knob, not a free win.** The auxiliary LM loss helps most on larger labeled sets; on tiny tasks it can be neutral or slightly hurt. Don't cargo‑cult `λ=0.5` without checking.
- **Tokenization mismatch.** Fine‑tuning data must be tokenized with the *exact* BPE used in pre‑training. A different tokenizer means the embeddings index a different vocabulary — garbage in.
- **Catastrophic forgetting / LR too high.** Fine‑tuning is meant to *nudge* pre‑trained weights. Too large a learning rate wipes out the transferable features; GPT‑1 used a small LR with warmup and a short schedule (few epochs).
- **Order sensitivity in similarity tasks.** Sentence pairs where order shouldn't matter (similarity) are processed in *both* orderings and summed — a single ordering bakes in spurious asymmetry.
- **It's TensorFlow 1.x.** The original repo won't run on modern TF without surgery. Study it for ideas; don't try to `pip install` your way into production with it.

## 7. When to Use vs Alternatives

| You want to… | Use | Why |
|---|---|---|
| **Understand** the origin of "pre‑train then fine‑tune" | **finetune‑transformer‑lm (this)** | The canonical GPT‑1 code; small, legible, historically pivotal. |
| **Fine‑tune an LLM today** (full or head‑only) | **🤗 `transformers` `Trainer`** | Modern, maintained, every architecture; automates exactly this recipe. |
| **Fine‑tune cheaply** (1 GPU, big model) | **PEFT / LoRA / QLoRA** | Train ~0.1–1% of params; same downstream quality at a fraction of memory. See the [`lora-controlnet`](./lora-controlnet.ipynb), [`qlora`](./qlora.ipynb) notebooks. |
| **Bidirectional understanding** (NLU) | **BERT‑style encoders** | Masked‑LM pre‑training sees both directions; often stronger on pure classification/extraction. |
| **No training at all** | **Few‑shot / prompting / RAG** | Modern large models often match fine‑tuning via in‑context examples. See [`few-shot-learning`](./few-shot-learning.ipynb), [`prompt-engineering`](./prompt-engineering.ipynb), [`rag`](./rag.ipynb). |
| **Build a GPT from scratch to learn** | **[minGPT](./mingpt.ipynb)** | The architecture this paper fine‑tunes, in ~300 readable lines. |

**Honest trade‑off:** GPT‑1's full fine‑tuning updates *all* weights — accurate but storage‑heavy (one full model copy per task) and prone to forgetting. Today, parameter‑efficient methods (LoRA/QLoRA) or plain prompting usually win on cost. The enduring contribution of `finetune-transformer-lm` is the *paradigm*, not the code.

## 8. Resources

- **Paper — "Improving Language Understanding by Generative Pre-Training" (GPT‑1)**: <https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf>
- **Original code repo (`openai/finetune-transformer-lm`)**: <https://github.com/openai/finetune-transformer-lm>
- **OpenAI blog announcement**: <https://openai.com/research/language-unsupervised>
- **🤗 Transformers — fine‑tuning guide (the modern way)**: <https://huggingface.co/docs/transformers/training>
- **The Illustrated GPT‑2 (Jay Alammar)** — visual intuition for the decoder stack: <https://jalammar.github.io/illustrated-gpt2/>
- **"Attention Is All You Need"** — the Transformer this builds on: <https://arxiv.org/abs/1706.03762>